# Felix synthetic test dataset — how it was made

This single notebook **documents and reproduces** the synthetic *E. coli* expression
dataset used to test the Felix pipeline, and writes the one file to model on:
**`felix_sim_testdata.csv`**.

**Why synthetic.** It is ground-truth data: the rules linking regulatory sequence to
expression are *known by construction*, so a method can be validated before any real
data. (It is therefore a method-validation testbed, not a real-accuracy benchmark.)

**How the data is generated (4 ideas).**
1. **Sequences.** Each construct is a promoter + 5′UTR + CDS + terminator, built by
   implanting mutated consensus motifs (σ70 −35 `TTGACA` / −10 `TATAAT`, Shine–Dalgarno,
   a terminator hairpin) into random background.
2. **Expression.** A transcript-level biophysical model (additive in log-space): the
   promoter dominates mRNA level, with secondary 5′-structure (ViennaRNA folding) and
   terminator stability terms. Three **tiers** of complexity — **A** additive, **B**
   `+` cross-region interaction, **C** `+` context-dependence — so model selection can be
   tested. **Tier B is the default** (most realistic).
3. **Realistic shape & noise.** Gaussian noise calibrated to a target replicate R²; a
   small fraction of genes are transcriptionally **silent** (a basal floor), giving the
   left-skewed distribution real RNA-seq shows. *Built from generic RNA-seq priors — not
   fit to any real dataset.*
4. **Homology & the split.** Real regulatory regions cluster (operons share a promoter;
   paralogs are similar), so we inject families and cluster them with **MMseqs2**. The
   provided `split`/`cluster_id` is at the **cluster** level — using it is essential
   (a random split leaks near-duplicates across train/test and inflates accuracy).

**Requires:** numpy, pandas, scikit-learn, ViennaRNA (`RNA`), and `mmseqs` on PATH.

## The engine (inlined — identical to the validated `felix` source)

In [ ]:
from __future__ import annotations
import json, os, subprocess, tempfile
from dataclasses import asdict, dataclass, field, replace
from pathlib import Path
import numpy as np
import pandas as pd
import RNA  # ViennaRNA

def five_prime_mfe(seq, window=50):
    """Minimum free energy (kcal/mol) of the first `window` nt (Kudla-2009 5' folding)."""
    s = str(seq).strip().upper().replace("U", "T")[:window]
    if not s:
        return 0.0
    _, mfe = RNA.fold(s)
    return float(mfe)


"""Holdout-safe reference profiling for the simulator.

The simulator may be anchored to *aggregate, marginal* properties of the real data
(the location/scale of the label) WITHOUT tapping the cross-validation fallacy — but
only if two rules hold (felix_plan.md:208; ESL ch7.10):

  1. Never borrow the CONDITIONAL map (which sequence -> which expression). That is the
     quantity the final-test holdout protects; the simulator must never see it. (The
     simulator generates its own sequence->expression map, so this holds by construction.)
  2. Any statistic borrowed from the real data is computed on the TRAIN partition only,
     AFTER the holdout clusters are defined. Profiling the full table (holdout included)
     lets the holdout's properties leak into every simulator-driven modeling decision
     (optimistic selection) — a weak but real form of the ch7.10 fallacy.

This module enforces rule 2 in code: `profile_reference` REFUSES to summarise values
without a split that excludes the holdout. The simulator's DEFAULT config borrows
nothing from PRECISE-1K — it uses generic RNA-seq priors (`GENERIC_RNASEQ_PRIOR`) and a
biological floor *mechanism*, so it is leakage-free by construction. A holdout-safe
`ReferenceProfile` is an optional refinement, available once the real regulatory FASTA
is downloaded, clustered (felix.data_prep), and split.
"""

from dataclasses import dataclass

import numpy as np
import pandas as pd


@dataclass(frozen=True)
class ReferenceProfile:
    """Aggregate, marginal summary used to calibrate the simulator's LOCATION/SCALE.
    Holds no per-sequence information, so it cannot carry the conditional map."""
    mean: float
    std: float
    skew: float
    floor_frac: float      # fraction below `floor_threshold` (the silent-gene floor)
    floor_threshold: float
    n: int
    source: str            # provenance, e.g. "PRECISE-1K train split" or "generic-prior"


# Generic RNA-seq priors — defensible from transcriptomics in general, fit to NO dataset:
#  - normalized log-expression is conventionally centered (mean 0) with ~unit-ish spread;
#  - RNA-seq has a detection floor + a silent-gene population (a few % to ~10%);
#  - the basal-vs-regulated gap is a couple of log-units.
# `skew` is left NaN on purpose: it is EMERGENT from the floor mechanism, never set.
GENERIC_RNASEQ_PRIOR = ReferenceProfile(
    mean=0.0, std=0.5, skew=float("nan"),
    floor_frac=0.07, floor_threshold=-2.0, n=0, source="generic-prior",
)


def profile_reference(values, split, *, partition: str = "train",
                      floor_threshold: float = -2.0,
                      source: str = "real-data") -> ReferenceProfile:
    """Summarise a real label distribution for simulator calibration — HOLDOUT-SAFE.

    Parameters
    ----------
    values : array-like per-row label (e.g. mean_log_tpm).
    split  : array-like per-row split assignment aligned to `values`
             ('train' / 'val' / 'test'). REQUIRED — passing None raises, because
             profiling without excluding the final-test holdout leaks it into model
             selection (ESL ch7.10; felix_plan.md:208).
    partition : rows to profile — 'train' (default) or 'trainval'. 'test' is refused.
    """
    if split is None:
        raise ValueError(
            "profile_reference() requires a split so the final-test holdout is excluded "
            "before profiling. Define the holdout clusters FIRST (felix.data_prep), then "
            "pass the split and profile partition='train'. (felix_plan.md:208; ESL ch7.10)")
    if "test" in str(partition):
        raise ValueError("refusing to profile the held-out 'test' partition")
    keep = {"train": ["train"], "trainval": ["train", "val"]}.get(partition)
    if keep is None:
        raise ValueError(f"partition must be 'train' or 'trainval', got {partition!r}")

    values = pd.Series(np.asarray(values, dtype=float)).reset_index(drop=True)
    split = pd.Series(np.asarray(split)).reset_index(drop=True)
    v = values[split.isin(keep)]
    if len(v) == 0:
        raise ValueError(f"no rows in partition {partition!r}")
    return ReferenceProfile(
        mean=float(v.mean()), std=float(v.std()), skew=float(v.skew()),
        floor_frac=float((v < floor_threshold).mean()), floor_threshold=floor_threshold,
        n=int(len(v)), source=source)


"""Controlled synthetic expression data with known ground truth.

See docs/sim_design.md for the literature-calibrated design. In brief: expression
is transcription x translation, each a thermodynamic rate ~ exp(-beta*dG); regions
are built by implanting mutated consensus motifs (sigma70 -35/-10, Shine-Dalgarno)
into i.i.d. background; three complexity tiers (A additive, B +cross-region
interaction, C context-dependent) match the function-class ladder; a Gaussian-on-log
noise model calibrated to a target replicate R^2 yields observable mean+variance.

This is METHOD-VALIDATION data, not an O1 result: we generate from features and a
later model predicts from related features, which is circular by construction. Its
value is that the rules are known, so O2 rule-extraction can be scored against truth
and O1 model selection can be checked for recovering the generating tier.

Literature anchors (see docs/sim_design.md for full cites): Promoter Calculator
additive dG (LaFleur/Salis 2022); -35/-10 ~74% + -10x-35 interaction (Urtecho 2018);
5' folding dominates translation (Kudla 2009); CAI~0 (Cambray 2018); Bintu F_reg
multiplicative cross-region; replicate R^2~0.95 (Urtecho) to ~0.99 (PRECISE-1K).
"""

import json
from dataclasses import asdict, dataclass, field, replace
from pathlib import Path

import numpy as np
import pandas as pd


BASES = "ACGT"

# Consensus motifs (the injected ground truth). Sequences a discovery tool should
# recover (sigma70 boxes, Shine-Dalgarno).
CONSENSUS = {
    "minus35": "TTGACA",
    "minus10": "TATAAT",
    "sd": "AGGAGG",
    # rho-independent terminator proxy: GC stem + U-tract. Affects mRNA abundance
    # (termination / 3' stability) — a transcript-level determinant.
    "terminator": "GGGGCCCCTTTT",
}

TIERS = ("A", "B", "C")


@dataclass
class SimParams:
    """All generative knobs, calibrated for a TRANSCRIPT-LEVEL readout (mRNA
    abundance, matching PRECISE-1K RNA-seq). Effect-size *proportions* follow the
    literature: transcription (promoter) dominates mRNA level; terminator / 3' and
    5' structure are secondary stability effects; the RBS/SD is a translation
    determinant and so is near-null at the transcript level (kept as a recoverable
    weak/negative-control motif for O2); CAI ~0 (Cambray 2018). Absolute scales are
    free knobs (literature reports relative rates only — no numeric beta)."""
    # promoter — dominant transcription-initiation term (Urtecho 2018: -35/-10 ~74%)
    a_minus35: float = 2.0
    a_minus10: float = 2.0
    spacer_opt: int = 17
    spacer_penalty: float = 0.30      # per bp deviation from optimum
    # terminator — moderate mRNA-stability/termination effect (heuristic; lit gap)
    a_term: float = 1.0
    # 5' folding — modest stability effect at transcript level; mfe<0 lowers level
    a_fold: float = 0.08              # per kcal/mol
    # RBS/SD — near-null at transcript level (translation determinant); kept small
    a_sd: float = 0.2
    # ~0 effect by default (Cambray 2018); present for the elongation-limited regime
    a_cai: float = 0.0
    # cross-region (Tier B)
    g_interact: float = 2.0          # -10 x -35 synergy (intra-promoter; Urtecho)
    g_cross: float = 0.8             # 5'-structure x terminator stability synergy
    # context dependence (Tier C): terminator matters more when 5' is unstable
    accept_mfe0: float = -15.0       # kcal/mol midpoint
    accept_steep: float = 0.30
    intercept: float = 2.0
    # sequence layout (nt)
    promoter_bg: int = 12            # background flanking the -35/-10 cassette
    utr5_len: int = 30
    cds_len: int = 90                # multiple of 3
    term_len: int = 30
    sd_to_start: int = 7             # SD..start spacing
    fold_window: int = 40            # 5' window for RNA.fold
    # noise
    target_replicate_r2: float = 0.95
    n_replicates: int = 3
    heteroscedastic: bool = False    # flag only — not literature-confirmed
    clip_range: tuple[float, float] | None = None  # detection floor/ceiling on log_expr

    # --- PRECISE-1K matching knobs (all default OFF -> base behaviour unchanged) ---
    # The simulator's *absolute* scale is an explicitly-declared free knob
    # (sim_design.md: "beta / absolute scales are free knobs"); the literature pins
    # only the relative effect-size proportions. So we may affine-map the latent
    # signal onto PRECISE-1K's log-TPM location/scale without touching any biology:
    # a common scale factor multiplies every effect equally, leaving all ratios fixed.
    calibrate_mean: float | None = None   # target mean of the latent signal (log-TPM)
    calibrate_std: float | None = None    # target std; both None => no calibration
    # PRECISE-1K's per-gene mean-expression distribution is LEFT-skewed with a
    # low-expression shoulder/floor (many genes are transcriptionally silent across
    # most conditions). We reproduce that shape — not via an ad-hoc clip, but
    # mechanistically — by building a fraction of constructs with an ablated promoter
    # (no recognisable -35/-10), which fall into a distinct low-expression mode.
    # `silenced_offset` (log-units, negative) drops those genes to basal/leaky
    # transcription well below the regulated range, producing the thin deep LEFT tail
    # (negative skew + floor) seen in PRECISE-1K rather than a mere shoulder bump.
    silenced_frac: float = 0.0
    silenced_offset: float = 0.0
    # Homology structure. Real E. coli regulatory regions are NOT i.i.d.: operon
    # co-members share one promoter (=> identical regulatory sequence) and paralogous
    # genes carry diverged-but-similar regulation. That homology is exactly why
    # MMseqs2 clustering + cluster-level splitting is load-bearing (felix_plan.md:121).
    # We inject it so the synthetic data reproduces the real cluster structure (and so
    # the sequence-vs-cluster-split leakage gap is demonstrable on ground truth).
    family_frac: float = 0.0              # P(a founder seeds a homologous family)
    family_size_mean: float = 2.5         # mean members per seeded family (founder + extras)
    # per-member divergence from the founder, drawn ~U(lo, hi); modelled as i.i.d.
    # per-site nucleotide substitution (Jukes-Cantor JC69: each mutated site -> one of
    # the other 3 bases uniformly). 0.0 == operon-identical (100% id); ~0.18 ~= paralog
    # (~82% id), comfortably inside the 70% MMseqs2 clustering threshold.
    family_divergence: tuple[float, float] = (0.0, 0.18)


def rnaseq_like_params(reference=None, **overrides) -> "SimParams":
    """SimParams for an RNA-seq-LIKE synthetic distribution, with leakage discipline.

    By default this borrows NOTHING from PRECISE-1K (see felix.reference): the
    location/scale come from generic RNA-seq priors and the left-skewed tail is a
    biological *mechanism* (a silent-gene floor), so the resulting skew is EMERGENT,
    never fit to a real moment. This sidesteps the cross-validation fallacy entirely —
    no real statistic (least of all a holdout one) informs any knob.

    Pass a holdout-safe `reference` (felix.reference.ReferenceProfile, built by
    profile_reference() on the TRAIN split only) to *refine* location/scale and the
    floor fraction from real data without ever touching the holdout. The homology
    knobs are sequence-structure priors, re-tuned to the real clustered TRAIN FASTA
    once it exists.
    """
    ref = reference or GENERIC_RNASEQ_PRIOR
    base = SimParams(
        # location/scale: generic prior by default; holdout-safe reference if supplied
        calibrate_mean=ref.mean,
        calibrate_std=ref.std,
        # floor MECHANISM (silent genes): fraction from prior/reference, basal gap a
        # documented prior (~2 log-units) — NOT tuned to match any target skew.
        silenced_frac=ref.floor_frac,
        silenced_offset=-2.0,
        # homology (sequence-space, not a label) — a structural prior, re-tuned to the
        # real clustered TRAIN regulatory FASTA when available.
        family_frac=0.45,
        family_size_mean=2.6,
        family_divergence=(0.0, 0.18),
    )
    return replace(base, **overrides)


# Back-compat alias. NB: the preset no longer FITS to PRECISE-1K — it uses generic
# RNA-seq priors (the old name predates the leakage fix). Prefer rnaseq_like_params.
precise1k_params = rnaseq_like_params


# ----------------------------------------------------------------------------
# sequence construction
# ----------------------------------------------------------------------------
def _rand_seq(rng, n):
    return "".join(rng.choice(list(BASES), size=n))


def _mutate(rng, consensus, n_mut):
    """Return a copy of `consensus` with exactly n_mut positions changed."""
    s = list(consensus)
    if n_mut > 0:
        pos = rng.choice(len(s), size=min(n_mut, len(s)), replace=False)
        for p in pos:
            s[p] = rng.choice([b for b in BASES if b != s[p]])
    return "".join(s)


def _match_fraction(site, consensus):
    return sum(a == b for a, b in zip(site, consensus)) / len(consensus)


def _mutate_seq(rng, seq: str, rate: float) -> str:
    """Apply i.i.d. per-site nucleotide substitution at `rate` (Jukes-Cantor JC69:
    each chosen site is replaced by one of the other 3 bases, uniformly). Used to
    spawn paralogous family members at a controlled divergence from a founder.
    No indels, so motif offsets are preserved and strengths can be re-derived."""
    if rate <= 0:
        return seq
    s = list(seq)
    hits = np.nonzero(rng.random(len(s)) < rate)[0]
    for i in hits:
        s[i] = rng.choice([b for b in BASES if b != s[i]])
    return "".join(s)


def _build_construct(rng, p: SimParams, silenced: bool = False):
    """Build one construct's regions + the latent strength variables.

    `silenced=True` ablates the promoter (-35/-10 fully randomised) so the construct
    has no functional transcription start -> a low-expression mode that reproduces
    PRECISE-1K's silent-gene floor/left-skew."""
    # sample motif strengths by choosing a mutation count (spread of 0..len). A
    # silenced promoter has BOTH boxes maximally mutated (match ~ 0) -> off.
    n35 = len(CONSENSUS["minus35"]) if silenced else int(rng.integers(0, len(CONSENSUS["minus35"]) + 1))
    n10 = len(CONSENSUS["minus10"]) if silenced else int(rng.integers(0, len(CONSENSUS["minus10"]) + 1))
    m35 = _mutate(rng, CONSENSUS["minus35"], n35)
    m10 = _mutate(rng, CONSENSUS["minus10"], n10)
    sd = _mutate(rng, CONSENSUS["sd"], rng.integers(0, len(CONSENSUS["sd"]) + 1))
    spacer_len = int(rng.integers(p.spacer_opt - 3, p.spacer_opt + 4))

    promoter = (
        _rand_seq(rng, p.promoter_bg) + m35
        + _rand_seq(rng, spacer_len) + m10
        + _rand_seq(rng, p.promoter_bg)
    )
    # 5'UTR: ...background... SD ...spacer... (start codon begins the CDS)
    utr_bg = max(p.utr5_len - len(sd) - p.sd_to_start, 0)
    utr5 = _rand_seq(rng, utr_bg) + sd + _rand_seq(rng, p.sd_to_start)
    cds = "ATG" + _rand_seq(rng, p.cds_len - 3)
    # terminator region with an embedded (mutated) terminator motif
    tcons = CONSENSUS["terminator"]
    tmot = _mutate(rng, tcons, rng.integers(0, len(tcons) + 1))
    term_bg = max(p.term_len - len(tmot), 0)
    term = _rand_seq(rng, term_bg) + tmot

    s35, s10, ssd, sterm = (
        _match_fraction(m35, CONSENSUS["minus35"]),
        _match_fraction(m10, CONSENSUS["minus10"]),
        _match_fraction(sd, CONSENSUS["sd"]),
        _match_fraction(tmot, tcons),
    )
    # 5' folding energy of the SD..start..CDS window (stability proxy), via ViennaRNA
    mfe = five_prime_mfe(utr5[-p.sd_to_start - len(sd):] + cds, window=p.fold_window)
    return {
        "promoter": promoter, "utr5": utr5, "cds": cds, "term": term,
        "spacer_len": spacer_len, "s35": s35, "s10": s10, "ssd": ssd,
        "sterm": sterm, "mfe": mfe,
        # motif offsets within each region (no indels) -> a derived family member can
        # re-extract the boxes after substitution and recompute its strengths.
        "m35_off": p.promoter_bg,
        "m10_off": p.promoter_bg + len(m35) + spacer_len,
        "sd_off": utr_bg,
        "tmot_off": term_bg,
        "tmot_len": len(tmot),
    }


def _derive_member(rng, parent: dict, p: SimParams, divergence: float) -> dict:
    """Spawn a homologous family member: the parent's regulatory regions diverged by
    JC69 substitution at `divergence`, carrying a *distinct* CDS (a different gene in
    the same operon/paralog family). Strengths are recomputed from the mutated boxes,
    so a paralog's regulation — and hence its expression — drifts with divergence,
    while an operon copy (divergence 0) is an exact regulatory duplicate."""
    prom = _mutate_seq(rng, parent["promoter"], divergence)
    utr5 = _mutate_seq(rng, parent["utr5"], divergence)
    term = _mutate_seq(rng, parent["term"], divergence)
    cds = "ATG" + _rand_seq(rng, p.cds_len - 3)

    l35, l10, lsd = len(CONSENSUS["minus35"]), len(CONSENSUS["minus10"]), len(CONSENSUS["sd"])
    m35 = prom[parent["m35_off"]: parent["m35_off"] + l35]
    m10 = prom[parent["m10_off"]: parent["m10_off"] + l10]
    sd = utr5[parent["sd_off"]: parent["sd_off"] + lsd]
    tmot = term[parent["tmot_off"]: parent["tmot_off"] + parent["tmot_len"]]
    mfe = five_prime_mfe(utr5[-p.sd_to_start - lsd:] + cds, window=p.fold_window)
    return {
        "promoter": prom, "utr5": utr5, "cds": cds, "term": term,
        "spacer_len": parent["spacer_len"],
        "s35": _match_fraction(m35, CONSENSUS["minus35"]),
        "s10": _match_fraction(m10, CONSENSUS["minus10"]),
        "ssd": _match_fraction(sd, CONSENSUS["sd"]),
        "sterm": _match_fraction(tmot, CONSENSUS["terminator"]),
        "mfe": mfe,
        "m35_off": parent["m35_off"], "m10_off": parent["m10_off"],
        "sd_off": parent["sd_off"], "tmot_off": parent["tmot_off"],
        "tmot_len": parent["tmot_len"],
    }


# ----------------------------------------------------------------------------
# latent expression by tier
# ----------------------------------------------------------------------------
def _spacer_term(p, spacer_len):
    return p.spacer_penalty * abs(spacer_len - p.spacer_opt)


def _sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def latent_log_expr(c: dict, p: SimParams, tier: str) -> tuple[float, dict]:
    """Return (log_expr_true, component breakdown) for one construct under a tier,
    at the TRANSCRIPT level (mRNA abundance).

    Tier A: additive region terms (promoter dominant; terminator + 5' structure
            secondary stability; SD near-null).
    Tier B: + (-10 x -35) promoter synergy and a 5'-structure x terminator
            stability synergy.
    Tier C: terminator's stabilizing effect is gated by 5' structure context —
            it matters more when the 5' end is unstable (context-dependent).
    """
    prom = p.a_minus35 * c["s35"] + p.a_minus10 * c["s10"] - _spacer_term(p, c["spacer_len"])
    fold = p.a_fold * c["mfe"]                       # mfe<0 -> lower mRNA level
    sd = p.a_sd * c["ssd"]                            # near-null at transcript level
    comp = {"intercept": p.intercept, "promoter": prom, "fold": fold, "sd": sd}

    if tier == "C":
        # 5' instability gate in (0,1): ~1 when mfe is high (unstable), ~0 when very
        # negative (well-protected) -> terminator stabilization matters more upstream-unstable
        gate = float(_sigmoid(p.accept_steep * (c["mfe"] - p.accept_mfe0)))
        term = p.a_term * c["sterm"] * gate
        comp["terminator"] = term
        comp["term_gate"] = gate
    else:
        comp["terminator"] = p.a_term * c["sterm"]

    log_expr = p.intercept + prom + fold + sd + comp["terminator"]

    if tier in ("B", "C"):
        inter = p.g_interact * c["s35"] * c["s10"]                  # intra-promoter
        # promoter x terminator super-additivity: a strongly-initiated transcript
        # benefits more from being well-terminated/stabilized (two distinct regions)
        cross = p.g_cross * ((c["s35"] + c["s10"]) / 2.0) * c["sterm"]
        comp["interact_35x10"] = inter
        comp["cross_prom_x_term"] = cross
        log_expr += inter + cross

    return float(log_expr), comp


# ----------------------------------------------------------------------------
# top-level simulate
# ----------------------------------------------------------------------------
@dataclass
class SimResult:
    labels: pd.DataFrame          # one row per construct (sequences + expr_mean/var + reps)
    truth: pd.DataFrame           # latent components per construct (ground truth)
    params: SimParams
    tier: str
    sigma: float                  # noise SD on log-expression


def simulate(n: int, tier: str = "A", seed: int = 0,
             params: SimParams | None = None) -> SimResult:
    """Generate `n` constructs under `tier`. Returns sequences, observable
    mean+variance, replicate draws, and the ground-truth component ledger."""
    if tier not in TIERS:
        raise ValueError(f"tier must be one of {TIERS}, got {tier!r}")
    p = params or SimParams()
    rng = np.random.default_rng(seed)

    # ---- build constructs: founders, their homologous families, and silenced genes.
    # A flat i.i.d. loop (the default, both knobs off) reproduces the original stream;
    # with family_frac>0 a founder may seed a family of near-duplicate members. ----
    built = []   # each: {"c": construct, "family": int, "role": str, "divergence": float}
    family_id = 0
    while len(built) < n:
        silenced = p.silenced_frac > 0 and rng.random() < p.silenced_frac
        founder = _build_construct(rng, p, silenced=silenced)
        built.append({"c": founder, "family": family_id, "role": "founder",
                      "divergence": 0.0, "silenced": silenced})
        if p.family_frac > 0 and rng.random() < p.family_frac:
            # number of EXTRA members ~ Geometric(success-prob 1/mean_extra), so
            # support is >=1 with mean = mean_extra (= family_size_mean - 1 founder).
            mean_extra = max(p.family_size_mean - 1.0, 1.0)
            n_extra = int(rng.geometric(1.0 / mean_extra))
            for _ in range(n_extra):
                if len(built) >= n:
                    break
                d = float(rng.uniform(*p.family_divergence))
                member = _derive_member(rng, founder, p, d)
                built.append({"c": member, "family": family_id, "role": "member",
                              "divergence": d, "silenced": silenced})
        family_id += 1
    built = built[:n]

    rows, truth_rows, latents = [], [], []
    for i, b in enumerate(built):
        c = b["c"]
        log_true, comp = latent_log_expr(c, p, tier)
        # silent genes fall to basal/leaky transcription (deep floor -> left tail)
        pen = p.silenced_offset if b["silenced"] else 0.0
        log_true += pen
        comp["silenced_penalty"] = pen
        latents.append(log_true)
        cid = f"sim{tier}_{i:05d}"
        rows.append({
            "construct_id": cid,
            "promoter": c["promoter"], "utr5": c["utr5"],
            "cds": c["cds"], "term": c["term"],
            "regulatory_seq": c["promoter"] + c["utr5"] + c["term"],
            "spacer_len": c["spacer_len"],
            "s35": c["s35"], "s10": c["s10"], "ssd": c["ssd"],
            "sterm": c["sterm"], "mfe": c["mfe"],
            # provenance (ground truth for homology / floor analyses)
            "family_id": f"fam{tier}_{b['family']:05d}",
            "role": b["role"], "divergence": b["divergence"], "silenced": b["silenced"],
        })
        truth_rows.append({"construct_id": cid, **comp, "log_expr_true": log_true})

    latents = np.asarray(latents)
    truth_df = pd.DataFrame(truth_rows)

    # ---- optional affine calibration of the latent signal onto PRECISE-1K's log-TPM
    # location/scale. A single scale `s` multiplies every additive component equally
    # (proportions preserved); the shift `b` is absorbed into the intercept so the
    # component ledger still sums to log_expr_true. Done BEFORE noise so the target
    # replicate-R^2 is set on the final scale. ----
    if p.calibrate_mean is not None and p.calibrate_std is not None:
        mu, sd0 = float(latents.mean()), float(latents.std()) or 1e-9
        s = p.calibrate_std / sd0
        b = p.calibrate_mean - s * mu
        latents = s * latents + b
        # term_gate is a (0,1) multiplier already baked into 'terminator' (not an
        # additive log term) and log_expr_true is reset directly -> exclude both.
        scale_cols = [col for col in truth_df.columns
                      if col not in ("construct_id", "term_gate", "log_expr_true")]
        truth_df[scale_cols] = truth_df[scale_cols] * s
        truth_df["intercept"] = truth_df["intercept"] + b
        truth_df["log_expr_true"] = latents
    # Calibrate homoscedastic noise to the target replicate R^2 (see sim_design.md).
    # For replicates each = true + eps, corr(rep1,rep2) = V/(V+sigma^2); the reported
    # replicate "R^2" is the coefficient of determination = corr^2, so target the
    # correlation at sqrt(R^2):  sigma^2 = V*(1-rho)/rho  with rho = sqrt(R^2).
    V = float(np.var(latents)) or 1e-9
    rho = float(np.sqrt(p.target_replicate_r2))
    sigma = float(np.sqrt(V * (1 - rho) / rho))

    df = pd.DataFrame(rows)
    rep_cols = []
    for r in range(p.n_replicates):
        if p.heteroscedastic:   # flag: more noise at low expression (count-like)
            scale = sigma * (1.5 - 0.5 * _sigmoid(latents - latents.mean()))
        else:
            scale = sigma
        draws = latents + rng.normal(0, 1, size=n) * scale
        if p.clip_range:
            draws = np.clip(draws, *p.clip_range)
        df[f"rep_{r}"] = draws
        rep_cols.append(f"rep_{r}")

    df["expr_mean"] = df[rep_cols].mean(axis=1)
    df["expr_var"] = (df[rep_cols].var(axis=1, ddof=1)
                      if p.n_replicates > 1 else sigma ** 2)
    df["tier"] = tier

    return SimResult(labels=df, truth=truth_df,
                     params=p, tier=tier, sigma=sigma)


def write_outputs(res: SimResult, outdir: Path) -> dict:
    """Write the same artifacts the real PRECISE-1K path consumes:
    data/raw/sim_<tier>.fasta (for data_prep clustering), plus labels + truth +
    generative params under data/interim/."""
    outdir = Path(outdir)
    raw, interim = outdir / "raw", outdir / "interim"
    raw.mkdir(parents=True, exist_ok=True)
    interim.mkdir(parents=True, exist_ok=True)
    t = res.tier

    fasta = raw / f"sim_{t}.fasta"
    with open(fasta, "w") as f:
        for _, row in res.labels.iterrows():
            f.write(f">{row['construct_id']}\n{row['regulatory_seq']}\n")

    labels_csv = interim / f"sim_{t}_labels.csv"
    res.labels.to_csv(labels_csv, index=False)
    truth_csv = interim / f"sim_{t}_truth.csv"
    res.truth.to_csv(truth_csv, index=False)
    params_json = interim / f"sim_{t}_params.json"
    params_json.write_text(json.dumps(
        {"tier": t, "sigma": res.sigma, "consensus": CONSENSUS,
         "params": asdict(res.params)}, indent=2))

    return {"fasta": fasta, "labels": labels_csv,
            "truth": truth_csv, "params": params_json}


def main(argv=None) -> int:
    import argparse
    ap = argparse.ArgumentParser(description="Felix synthetic data generator")
    ap.add_argument("--n", type=int, default=2000)
    ap.add_argument("--tier", choices=list(TIERS), default="A")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--replicates", type=int, default=3)
    ap.add_argument("--replicate-r2", type=float, default=0.95)
    ap.add_argument("--heteroscedastic", action="store_true")
    ap.add_argument("--rnaseq-like", "--match-precise1k", dest="rnaseq_like",
                    action="store_true",
                    help="use rnaseq_like_params(): generic-prior RNA-seq-like scale, "
                         "floor, and homology (operon/paralog) cluster structure "
                         "(--match-precise1k is a deprecated alias)")
    ap.add_argument("--outdir", type=Path, default=Path("data"))
    args = ap.parse_args(argv)

    if args.rnaseq_like:
        p = rnaseq_like_params(n_replicates=args.replicates,
                               target_replicate_r2=args.replicate_r2,
                               heteroscedastic=args.heteroscedastic)
    else:
        p = SimParams(n_replicates=args.replicates,
                      target_replicate_r2=args.replicate_r2,
                      heteroscedastic=args.heteroscedastic)
    res = simulate(args.n, tier=args.tier, seed=args.seed, params=p)
    paths = write_outputs(res, args.outdir)
    print(f"tier {args.tier}: {args.n} constructs, noise sigma={res.sigma:.3f} "
          f"(target replicate R^2={args.replicate_r2})")
    for k, v in paths.items():
        print(f"  {k}: {v}")
    return 0




# --- clustering (MMseqs2) + cluster-level split, inlined ----------------------------
def cluster_regions(fasta_path, min_seq_id=0.7, coverage=0.8):
    """Cluster regulatory regions at `min_seq_id` identity with MMseqs2 easy-linclust."""
    with tempfile.TemporaryDirectory() as td:
        pref = os.path.join(td, "clu")
        subprocess.run(["mmseqs", "easy-linclust", str(fasta_path), pref,
                        os.path.join(td, "tmp"), "--min-seq-id", str(min_seq_id),
                        "-c", str(coverage)], check=True, capture_output=True)
        df = pd.read_csv(pref + "_cluster.tsv", sep="\t", header=None,
                         names=["cluster_id", "seq_id"])
    return df[["seq_id", "cluster_id"]]

def assign_cluster_splits(clusters, fracs=(0.8, 0.1, 0.1), seed=0):
    """80/10/10 split assigned at the CLUSTER level (no cluster spans two splits)."""
    uniq = np.sort(clusters["cluster_id"].unique())
    sh = np.random.default_rng(seed).permutation(uniq)
    n = len(sh); n_tr = int(round(fracs[0] * n)); n_va = int(round(fracs[1] * n))
    tr, va = set(sh[:n_tr]), set(sh[n_tr:n_tr + n_va])
    out = clusters.copy()
    out["split"] = out["cluster_id"].map(
        lambda c: "train" if c in tr else ("val" if c in va else "test"))
    return out

## Generate the dataset and write `felix_sim_testdata.csv`

Deterministic (`seed=0`). Default tier **B**; change `TIER` to `"A"` or `"C"` for the
other rungs. Only *observable* columns are written — the generative latents are kept out
so the file is a fair test of predicting from sequence.

In [2]:
TIER, N = "B", 3923
res = simulate(N, tier=TIER, seed=0, params=rnaseq_like_params())
lab = res.labels.copy()

# Assemble the FULL transcription unit in genomic order
# (promoter -> 5'UTR -> CDS -> terminator) plus each part's length, so the parts can be
# CUT from `full_tu` the SAME way for synthetic and real data (see the final cell).
lab["full_tu"]      = lab.promoter + lab.utr5 + lab.cds + lab.term
lab["promoter_len"] = lab.promoter.str.len()
lab["utr5_len"]     = lab.utr5.str.len()
lab["cds_len"]      = lab.cds.str.len()
lab["term_len"]     = lab.term.str.len()

# Cluster on the REGULATORY regions only (promoter+5'UTR+terminator, no CDS): the split
# tests generalization to novel regulatory context (the plan clusters regulatory regions).
fasta = Path("_sim_regions.fasta")
fasta.write_text("".join(f">{r.construct_id}\n{r.regulatory_seq}\n" for r in lab.itertuples()))
splits = assign_cluster_splits(cluster_regions(fasta)); fasta.unlink()

model = lab.merge(splits.rename(columns={"seq_id": "construct_id"}), on="construct_id")
COLS = ["construct_id", "full_tu", "promoter_len", "utr5_len", "cds_len", "term_len",
        "promoter", "utr5", "cds", "term", "rep_0", "rep_1", "rep_2",
        "expr_mean", "expr_var", "cluster_id", "split", "tier"]
model[COLS].to_csv("felix_sim_testdata.csv", index=False)

print(f"wrote felix_sim_testdata.csv  ({len(model)} rows, tier {TIER})")
print("clusters:", model.cluster_id.nunique(), "| split:", model.split.value_counts().to_dict())
# sanity: parts re-cut from full_tu match the stored part columns
r0 = model.iloc[0]
assert r0.full_tu[:r0.promoter_len] == r0.promoter
assert r0.full_tu[r0.promoter_len:r0.promoter_len+r0.utr5_len] == r0.utr5
print("cut check: parts slice out of full_tu correctly")
model[["construct_id", "full_tu", "promoter_len", "utr5_len", "cds_len", "term_len", "expr_mean"]].head(3)

wrote felix_sim_testdata.csv  (3923 rows, tier B)
clusters: 2560 | split: {'train': 3114, 'val': 427, 'test': 382}
cut check: parts slice out of full_tu correctly


,construct_id,full_tu,promoter_len,utr5_len,cds_len,term_len,expr_mean
0,simB_00000,ATAGACCCCAAAGGGACGAGGGCGTCCTTTCGTTATGATGTGGCTA...,51,30,90,30,0.358518
1,simB_00001,GTGCAGAGATTACAAGGGTCATTAGTTCTTAAACAAACACCGCGGC...,54,30,90,30,-1.001671
2,simB_00002,GGTAAAGCAGTTTTGACACGTGACTATGCTTCGTGCTTTTAATTAG...,55,30,90,30,1.076665


## Why you must use the `split` column (don't split randomly)

Quick demonstration: the *same* memorization-prone model scored two ways — a random
split vs the provided cluster split. The random split looks better only because it leaks
near-identical homologs across train/test.

In [3]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import KFold, GroupKFold, cross_val_score
from itertools import product

KM = ["".join(p) for p in product("ACGT", repeat=4)]; KI = {k: i for i, k in enumerate(KM)}
def kmer4(s):
    v = np.zeros(256)
    for i in range(len(s) - 3):
        j = KI.get(s[i:i+4]);  v[j] += 1 if j is not None else 0
    return v / max(v.sum(), 1)

X = np.vstack([kmer4(s) for s in model.regulatory_seq]); y = model.expr_mean.values
knn = KNeighborsRegressor(5, metric="cosine")
r_rand = cross_val_score(knn, X, y, cv=KFold(5, shuffle=True, random_state=0), scoring="r2").mean()
r_clus = cross_val_score(knn, X, y, cv=GroupKFold(5), groups=model.cluster_id.values, scoring="r2").mean()
print(f"random  split CV R2 = {r_rand:.3f}   <- inflated by homolog leakage")
print(f"cluster split CV R2 = {r_clus:.3f}   <- the honest number")
print(f"leakage = {r_rand - r_clus:+.3f} R2")

random  split CV R2 = 0.171   <- inflated by homolog leakage
cluster split CV R2 = -0.091   <- the honest number
leakage = +0.262 R2


## How to use `felix_sim_testdata.csv`

Each row is **one full transcription unit with one expression value** (like a PRECISE-1K gene).

**Sequence.** `full_tu` is the complete TU in genomic order (promoter → 5′UTR → CDS →
terminator). Cut it into parts with the length columns — the *same* cut works on real data
once you extract a TU and its coordinates:

```python
def cut(r):
    s, p, u, c = r.full_tu, r.promoter_len, r.utr5_len, r.cds_len
    return dict(promoter=s[:p], utr5=s[p:p+u], cds=s[p+u:p+u+c], term=s[p+u+c:])
```

The pre-cut `promoter/utr5/cds/term` columns are included for convenience — they are
exactly these slices.

**Label.** `expr_mean` = expression to predict (log scale); `expr_var` = observed variance;
`rep_0..2` = replicate draws.

**Split.** `cluster_id` = MMseqs2 cluster of the *regulatory region* (the split unit);
`split` ∈ {train, val, test} assigned at the cluster level.

**The three rules**
1. Predict `expr_mean` from `full_tu` (or its parts).
2. Evaluate with `split`; for CV, **group by `cluster_id`** — never split rows randomly.
3. Use `split == "test"` exactly once, at the end.

Re-run this notebook top-to-bottom to regenerate the file (deterministic, `seed=0`).